In [ ]:
import os
import warnings
import logging
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_LOG_LEVEL"] = "3"
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/usr/local/cuda"
tf.get_logger().setLevel("ERROR")

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"

import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

import sys
sys.path.append('/kaggle/input/datasets/keithmarange/cnn-method-baseline-stuff/')
sys.path.append('/kaggle/input/cmi-competition-code')

import pandas as pd
import data_utils
import os
import utils
from sklearn.pipeline import Pipeline
from scipy.stats import randint
from skopt.space import Categorical, Integer
from sklearn_genetic.space import Categorical as ECat, Integer as EInt
from sklearn.model_selection import GridSearchCV
from sklearn_genetic import GASearchCV
from sklearn.model_selection import RandomizedSearchCV
from skopt import BayesSearchCV
from sklearn.model_selection import GroupKFold
from skopt.space import Categorical, Integer, Real
from sklearn_genetic.space import Categorical as ECat, Integer as EInt, Continuous as EFloat
from scipy.stats import randint, uniform, loguniform
from sklearn.metrics import f1_score, make_scorer
import numpy as np

from sklearn.model_selection import KFold
import importlib
warnings.filterwarnings('ignore', module='deap')

import utils
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
data_folder = data_utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
n_splits = 3
cv = GroupKFold(n_splits=n_splits)
scoring_metric = 'f1_macro'
model_target = 'gesture_action'

scoring = None

tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]
acc_cols = ['acc_x', 'acc_y', 'acc_z']

pipe_name = "temporal_extractor"
classifier_name = 'CNN_1D'

search_mode = "grid"  # grid, random, evolutionary, bayesian

pipe_name = 'temporal_extractor'
candidates = 3
generations = 2
tournament_size = 2
elitism = True
crossover_probability = 0.8
mutation_probability = 0.2

chosen_orientation = ['Seated Straight']

train_size = 0.5

In [ ]:
train_df = raw_train_df.set_index('row_id')

if train_size is None:
    rows = (train_demo_df['adult_child'] == 1) & (train_demo_df['sex'] == 1) & (train_demo_df['handedness'] == 1)
    ideal_subject_ids = train_demo_df.loc[rows].sort_values(by='elbow_to_wrist_cm', ascending=False)['subject'].to_list()

    train_sample_df = train_df.loc[train_df['subject'].isin(ideal_subject_ids), :]
    train_sample_df = train_sample_df[train_sample_df['sequence_type'] == 'Target']
    train_sample_df['gesture_action'] = train_sample_df['gesture'].str.split(' - ').str[-1]
    print(train_sample_df['sequence_id'].nunique())

elif train_size == 0:
    some_sequences = train_df['sequence_id'].unique()[10:]
    train_sample_df = train_df[train_df['sequence_id'].isin(some_sequences)]
    train_sample_df['gesture_action'] = train_sample_df['gesture'].str.split(' - ').str[-1]
    
else:
    target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

    target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
    target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]
    target_only_df = target_only_df[target_only_df['phase'] == 'Gesture']

    train_sample_df, test_sample_df = data_utils.sample_balanced_split(
        target_only_df,
        train_pct=train_size,
        test_pct=0.2
    )

if chosen_orientation is not None:
    train_sample_df = train_sample_df[train_sample_df['orientation'].isin(chosen_orientation)]

In [ ]:
if search_mode == "bayesian":
    param_space = {
        f"{pipe_name}__acc_mode": Categorical(["smoothed", "velocity", "displacement", "jerk"]),
        f"{pipe_name}__linear_acc_mode": Categorical(["baseline"]),
        f"{pipe_name}__use_acc_magnitude": Categorical([True]),
        f"{pipe_name}__use_linear_acc_magnitude": Categorical([True]),
        f"{pipe_name}__sampling_rate": Categorical([10, 20, 25, 50, 100]),
        f"{pipe_name}__compute_dt": Categorical([True]),
        f"{pipe_name}__clip_value": Categorical([50.0]),
        f"{pipe_name}__interp_mode": Categorical(["linear"]),
        f"{pipe_name}__use_highpass_fallback": Categorical([True]),
        f"{pipe_name}__window_size": Integer(2, 21),
        f"{pipe_name}__smooth_alpha": Real(0.0, 1.0),
        f"{pipe_name}__standardize": Categorical(["mean_std"]),
        f"{pipe_name}__include_mask": Categorical([False, True]),

        f"{pipe_name}__rotation_mode": Categorical(["quaternion", "euler", "delta_euler", "rot6d"]),
        f"{pipe_name}__fix_quaternion_sign": Categorical([True]),

        f"{pipe_name}__tof_mode": Categorical(["pooled", "pooled_diff", "sensor_stats", "pooled_stats"]),
        f"{pipe_name}__tof_fill_mode": Categorical(["nan_interpolate", "far_255", "far_500"]),
        f"{pipe_name}__thm_mode": Categorical(["diff", "centered", "centered_diff"]),

        f"{classifier_name}__maxlen": Integer(10, 200),
        f"{classifier_name}__padding_value": Categorical([-999.0]),
        f"{classifier_name}__conv_filters": Categorical(["64", "128", "32-64"]),
        f"{classifier_name}__kernel_sizes": Categorical(["5", "3-5-7"]),
        f"{classifier_name}__pool_sizes": Categorical(["2", "none-none",  "2-none-none"]),
        f"{classifier_name}__dense_units": Categorical(["none", "128-64"]),
        f"{classifier_name}__use_batch_norm": Categorical([True, False]),
        f"{classifier_name}__spatial_dropout": Real(0.0, 0.3),
        f"{classifier_name}__dropout": Real(0.0, 0.5),
        f"{classifier_name}__learning_rate": Real(1e-4, 2e-3, prior="log-uniform"),
        f"{classifier_name}__batch_size": Categorical([16, 32]),
        f"{classifier_name}__epochs": Integer(100, 150),
        f"{classifier_name}__patience": Integer(1, 20),
    }

elif search_mode == "evolutionary":
    param_space = {
        f"{pipe_name}__acc_mode": ECat(["raw", "smoothed", "velocity", "displacement", "jerk"]),
        f"{pipe_name}__linear_acc_mode": ECat([None, "baseline"]),
        f"{pipe_name}__use_acc_magnitude": ECat([False, True]),
        f"{pipe_name}__use_linear_acc_magnitude": ECat([False, True]),
        f"{pipe_name}__sampling_rate": ECat([20, 25, 50]),
        f"{pipe_name}__compute_dt": ECat([True]),
        f"{pipe_name}__clip_value": ECat([None, 20.0, 50.0, 100.0]),
        f"{pipe_name}__interp_mode": ECat([None, "linear"]),
        f"{pipe_name}__use_highpass_fallback": ECat([True]),
        f"{pipe_name}__window_size": EInt(3, 21),
        f"{pipe_name}__smooth_alpha": ECat([None, 0.2, 0.5, 0.8]),
        f"{pipe_name}__standardize": ECat([None, "mean_std"]),
        f"{pipe_name}__include_mask": ECat([False]),

        f"{pipe_name}__rotation_mode": ECat([None, "quaternion", "euler", "delta_euler", "angular_velocity", "rot6d"]),
        f"{pipe_name}__fix_quaternion_sign": ECat([True, False]),

        f"{pipe_name}__tof_mode": ECat([None, "pooled", "pooled_diff", "sensor_stats", "pooled_stats"]),
        f"{pipe_name}__tof_fill_mode": ECat(["nan_interpolate", "far_255", "far_500"]),
        f"{pipe_name}__thm_mode": ECat([None, "raw", "diff", "centered", "centered_diff"]),

        f"{classifier_name}__maxlen": EInt(16, 160),
        f"{classifier_name}__padding_value": ECat([-999.0]),
        f"{classifier_name}__conv_filters": ECat(["32", "64", "128", "32-64", "64-64", "64-128", "32-64-128", "64-128-128"]),
        f"{classifier_name}__kernel_sizes": ECat(["3", "5", "7", "3-3", "5-5", "3-5-7", "5-7-9"]),
        f"{classifier_name}__pool_sizes": ECat(["none", "2", "none-none", "2-none", "2-2", "none-none-none", "2-none-none", "2-2-none"]),
        f"{classifier_name}__dense_units": ECat(["none", "32", "64", "128", "64-32", "128-64"]),
        f"{classifier_name}__use_batch_norm": ECat([True, False]),
        f"{classifier_name}__spatial_dropout": EFloat(0.0, 0.3),
        f"{classifier_name}__dropout": EFloat(0.0, 0.5),
        f"{classifier_name}__learning_rate": EFloat(1e-4, 2e-3),
        f"{classifier_name}__batch_size": ECat([16, 32, 64]),
        f"{classifier_name}__epochs": ECat([60, 80, 120]),
        f"{classifier_name}__patience": ECat([8, 12, 20]),
    }

elif search_mode == "random":

    param_space = {
        f"{pipe_name}__acc_mode": ["raw", "smoothed", "velocity", "displacement", "jerk"],
        f"{pipe_name}__linear_acc_mode": [None, "baseline"],
        f"{pipe_name}__use_acc_magnitude": [False, True],
        f"{pipe_name}__use_linear_acc_magnitude": [False, True],
        f"{pipe_name}__sampling_rate": [20, 25, 50],
        f"{pipe_name}__compute_dt": [True],
        f"{pipe_name}__clip_value": [None, 20.0, 50.0, 100.0],
        f"{pipe_name}__interp_mode": [None, "linear"],
        f"{pipe_name}__use_highpass_fallback": [True],
        f"{pipe_name}__window_size": randint(3, 22),
        f"{pipe_name}__smooth_alpha": [None, 0.2, 0.5, 0.8],
        f"{pipe_name}__standardize": [None, "mean_std"],
        f"{pipe_name}__include_mask": [False],

        f"{pipe_name}__rotation_mode": [None, "quaternion", "euler", "delta_euler", "angular_velocity", "rot6d"],
        f"{pipe_name}__fix_quaternion_sign": [False, True],

        f"{pipe_name}__tof_mode": [None, "pooled", "pooled_diff", "sensor_stats", "pooled_stats"],
        f"{pipe_name}__tof_fill_mode": ["nan_interpolate", "far_255", "far_500"],
        f"{pipe_name}__thm_mode": [None, "raw", "diff", "centered", "centered_diff"],

        f"{classifier_name}__maxlen": randint(16, 161),
        f"{classifier_name}__padding_value": [-999.0],
        f"{classifier_name}__conv_filters": ["32", "64", "128", "32-64", "64-64", "64-128", "32-64-128", "64-128-128"],
        f"{classifier_name}__kernel_sizes": ["3", "5", "7", "3-3", "5-5", "3-5-7", "5-7-9"],
        f"{classifier_name}__pool_sizes": ["none", "2", "none-none", "2-none", "2-2", "none-none-none", "2-none-none", "2-2-none"],
        f"{classifier_name}__dense_units": ["none", "32", "64", "128", "64-32", "128-64"],
        f"{classifier_name}__use_batch_norm": [True, False],
        f"{classifier_name}__spatial_dropout": uniform(0.0, 0.3),
        f"{classifier_name}__dropout": uniform(0.0, 0.5),
        f"{classifier_name}__learning_rate": loguniform(1e-4, 2e-3),
        f"{classifier_name}__batch_size": [16, 32, 64],
        f"{classifier_name}__epochs": [60, 80, 120],
        f"{classifier_name}__patience": [8, 12, 20],
    }

elif search_mode == "grid":
    
    param_space = {
        f"{pipe_name}__acc_mode": ["jerk"],
        f"{pipe_name}__linear_acc_mode": [None],
        f"{pipe_name}__use_acc_magnitude": [False],
        f"{pipe_name}__use_linear_acc_magnitude": [False],
        f"{pipe_name}__sampling_rate": [20],
        f"{pipe_name}__compute_dt": [True],
        f"{pipe_name}__clip_value": [40],
        f"{pipe_name}__interp_mode": [None],
        f"{pipe_name}__use_highpass_fallback": [False],
        f"{pipe_name}__window_size": [4],
        f"{pipe_name}__smooth_alpha": [None],
        f"{pipe_name}__standardize": [None],
        f"{pipe_name}__include_mask": [False],

        f"{pipe_name}__rotation_mode": ["angular_velocity"],
        f"{pipe_name}__fix_quaternion_sign": [True],

        f"{pipe_name}__tof_mode": ["pooled"],
        f"{pipe_name}__tof_fill_mode": ["far_255"],
        f"{pipe_name}__thm_mode": ["centered_diff"],

        f"{classifier_name}__maxlen": [70],
        f"{classifier_name}__padding_value": [-999.0],
        f"{classifier_name}__conv_filters": ["64-128-128"],
        f"{classifier_name}__kernel_sizes": ["3-3"],
        f"{classifier_name}__pool_sizes": ["2-2"],
        f"{classifier_name}__dense_units": ["64-32"],
        f"{classifier_name}__use_batch_norm": [True],
        f"{classifier_name}__spatial_dropout": [0.1],
        f"{classifier_name}__dropout": [0.2],
        f"{classifier_name}__learning_rate": [5e-4],
        f"{classifier_name}__batch_size": [32],
        f"{classifier_name}__epochs": [50],
        f"{classifier_name}__patience": [2],
    }

In [ ]:
importlib.reload(utils)         

pipeline = Pipeline([
    (pipe_name, utils.SequenceExtractor()),
    (classifier_name, utils.KerasCNN1DSequenceClassifier(
        target=model_target
    )),
])

if search_mode == "bayesian":
    search_obj = BayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=candidates,
        scoring=scoring,
        cv=cv,
        n_jobs=1,
        verbose=3,
        random_state=42,
        refit=True,
        return_train_score=True,
        error_score=np.nan
    )

elif search_mode == "random":
    search_obj = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_space,
        n_iter=candidates,
        scoring=scoring,
        cv=cv,
        n_jobs=1,
        verbose=3,
        random_state=42,
        refit=True,
        return_train_score=True,
        error_score = np.nan
    )

elif search_mode == "evolutionary":
    cv = KFold(n_splits=3, shuffle=True, random_state=42)
    
    search_obj = GASearchCV(
        estimator=pipeline,
        cv=cv,
        scoring=scoring,
        param_grid=param_space,
        population_size=candidates,
        generations=generations,
        tournament_size=tournament_size,
        elitism=elitism,
        crossover_probability=crossover_probability,
        mutation_probability=mutation_probability,
        criteria="max",
        n_jobs=1,
        verbose=True,
        keep_top_k=5,
        return_train_score=True,
        error_score=np.nan
    )

elif search_mode == "grid":
    search_obj = GridSearchCV(
        estimator=pipeline,
        param_grid=param_space,
        scoring=scoring,
        cv=cv,
        n_jobs=1,
        verbose=4,
        refit=True,
        return_train_score=True,
        error_score=np.nan
    )

else:
    raise ValueError("search_mode must be one of: 'bayesian', 'random', 'evolutionary', 'grid'")

In [ ]:
y = train_sample_df[['sequence_id', model_target]]
groups = train_sample_df['sequence_id']

print(f"--- {search_mode} Search ---")
if search_mode == 'evolutionary':
    search_obj.fit(train_sample_df, y)
else:
    search_obj.fit(train_sample_df, y, groups=groups)

In [ ]:
# --- 1. Model Prediction & Evaluation ---
best_model = search_obj.best_estimator_
X_test = test_sample_df.copy()

# Get unique ground truth labels per sequence
y_true_seq = (test_sample_df[['sequence_id', model_target]]
              .drop_duplicates('sequence_id')
              .reset_index(drop=True))

y_pred_seq = best_model.predict(X_test)

# Calculate Accuracy
test_accuracy = accuracy_score(y_true_seq[model_target], y_pred_seq)

print(f"--- Final Test Results ---")
print(f"Best CV Score: {search_obj.best_score_:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:\n", classification_report(y_true_seq[model_target], y_pred_seq))

# --- 2. Cleanly Append Test Results to CV Results ---
if hasattr(search_obj, 'cv_results_'):
    # Convert search results to DataFrame
    cv_results_df = pd.DataFrame(search_obj.cv_results_)
    cv_results_df['search_mode'] = search_mode
    cv_results_df['target'] = model_target
    
    # Create a "Final Test" row matching the CV columns
    # We use 'params' to label it and put the accuracy in 'mean_test_score'
    test_result_row = pd.DataFrame({
        'params': ['FINAL_HOLD_OUT_TEST'],
        'mean_test_score': [test_accuracy],
        'std_test_score': [0],
        'rank_test_score': [0]
    })
    
    # Concat results - holes in the table (like split scores) fill with NaN
    final_report_df = pd.concat([cv_results_df, test_result_row], ignore_index=True)
    
    # Save the consolidated report
    file_path = f"{model_run_folder_name}{search_mode}_{classifier_name}_results.csv"
    final_report_df.to_csv(file_path, index=False)
    
    print(f"Results consolidated and saved to: {file_path}")